# BraTS-PEDs 2026 — SwinUNETR: **ET Subregion** (Binary: BG vs ET)
**Label map (binary):** `0`=Background · `1`=ET (Enhancing Tumour, original label 1)

**Architecture choices for ET:**
- **T1c (T1CE) as PRIMARY modality** — active tumour rim enhances brightly post-contrast
- **SwinUNETR** — shifted-window self-attention for long-range context across the ring-enhancing boundary
- **Combined Loss: Focal Tversky Loss (FTL) + FocalDice + WeightedBCE** — triple-term loss for extreme imbalance and hard boundary focus
- **FTL α=0.3, β=0.7, γ=0.75** — penalises False Negatives heavily (missing ET is worse than over-prediction)
- **FocalDice γ=3.0** — concentrates gradient on uncertain voxels at tumour rim
- **WeightedBCE pos_weight=30** — prevents trivial all-background predictions
- **T1c foreground crop + pos:neg=4:1** — patches forced onto enhancing voxels
- **Gamma correction on T1c/T1n** — robust to contrast-uptake intensity variability

Run all cells top to bottom. Only edit **Cell 2 — Configuration**.


## Cell 1 — Imports

In [1]:

import os, time, random, warnings
from pathlib import Path
from datetime import datetime
from functools import partial
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast

torch.backends.cudnn.benchmark = True

import logging
torch._logging.set_logs(recompiles=False)
logging.getLogger("torch._dynamo").setLevel(logging.ERROR)

from monai.config import print_config
from monai.utils import set_determinism
from monai.utils.enums import MetricReduction
from monai.data import (
    SmartCacheDataset, DataLoader, Dataset,
    decollate_batch, pad_list_data_collate,
)
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    NormalizeIntensityd, CropForegroundd,
    RandCropByPosNegLabeld, RandFlipd, RandRotate90d,
    RandShiftIntensityd, RandScaleIntensityd, RandGaussianNoised,
    RandAdjustContrastd, RandGaussianSmoothd,
    Activations, AsDiscrete, MapTransform,
)
from monai.networks.nets import UNet, SwinUNETR
from monai.networks.layers import Norm
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.inferers import sliding_window_inference

NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print_config()
print(f"\nPyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"GPUs found: {NUM_GPUS}")
for i in range(NUM_GPUS):
    vram = round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({vram} GB)")


MONAI version: 1.5.2
Numpy version: 1.26.4
Pytorch version: 2.4.1+cpu
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: d18565fb3e4fd8c556707f91ac280a2dc3f681c1
MONAI __file__: C:\Users\<username>\anaconda3\envs\brats\lib\site-packages\monai\__init__.py

Optional dependencies:
Pytorch Ignite version: NOT INSTALLED or UNKNOWN VERSION.
ITK version: NOT INSTALLED or UNKNOWN VERSION.
Nibabel version: 5.4.2
scikit-image version: 0.25.2
scipy version: 1.15.3
Pillow version: 12.2.0
Tensorboard version: NOT INSTALLED or UNKNOWN VERSION.
gdown version: NOT INSTALLED or UNKNOWN VERSION.
TorchVision version: 0.19.1+cpu
tqdm version: 4.67.3
lmdb version: NOT INSTALLED or UNKNOWN VERSION.
psutil version: 7.2.2
pandas version: 2.3.3
einops version: 0.8.2
transformers version: NOT INSTALLED or UNKNOWN VERSION.
mlflow version: NOT INSTALLED or UNKNOWN VERSION.
pynrrd version: NOT INSTALLED or UNKNOWN VERSION.
clearml version: NOT INSTALLED or UNKNOWN VERSION.

For

## Cell 2 — Configuration
**Edit the two paths. Everything else is pre-tuned for ET subregion.**


In [12]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  EDIT THESE PATHS                                           ║
# ╚══════════════════════════════════════════════════════════════╝
DATA_ROOT        = Path(r"C:\Users\BRAVO 15\OneDrive\Desktop\Intern\model_code\BraTS_2026\All_data_Deskullp\Test")
EXPERIMENTS_ROOT = Path(r"C:\Users\BRAVO 15\OneDrive\Desktop\Intern\model_code\BraTS_2026")

MODEL_NAME = "SwinUNETR_ET_BinarySubregion"
MODEL_ROOT = EXPERIMENTS_ROOT / MODEL_NAME
RUN_NAME   = f"run_{datetime.now():%Y%m%d_%H%M}"
RUN_DIR    = MODEL_ROOT / RUN_NAME
for sub in ["", "figures", "predictions", "checkpoints"]:
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
print("Run directory:", RUN_DIR)

SEED = 42
set_determinism(seed=SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

SPLIT_RATIOS = (0.80, 0.20)

# ── ET-specific: T1c is the PRIMARY contrast source ───────────────────────────
# T1c  = T1-weighted post-contrast: active enhancing tumour is BRIGHT
# T1n  = pre-contrast: provides baseline for enhancement ratio
# T2w  = peritumoral context (complementary, not primary for ET)
# T2f  = FLAIR: helps delineate tumour extent vs. oedema
IMAGE_KEYS = ["t1n", "t1c", "t2w","t2f"]   # all 4 modalities as input channels
ET_KEYS    = ["t1c", "t1n"]                    # T1CE + T1n — primary ET contrast pair
ALL_KEYS   = IMAGE_KEYS + ["label"]

# Binary task: BG=0, ET=1
# Label remapping: original label 1 (ET) → 1; everything else → 0
IN_CHANNELS  = 4
OUT_CHANNELS = 2

# ── Patch & inference ─────────────────────────────────────────────────────────
# 96³ patch: ET is typically ~1000-5000 voxels — larger than CC, smaller than ED
# Larger patch captures the full ring-enhancing boundary context
PATCH_SIZE          = (64, 64, 64)
SW_ROI              = (64, 64, 64)  # Full tumour context at inference
SW_BATCH            = 2
SW_OVERLAP          = 0.50             # Higher overlap = better ring boundary coverage
NUM_POS_NEG_SAMPLES = 4                # Patches per case per iteration

# ── SwinUNETR architecture ────────────────────────────────────────────────────
# feature_size=48: standard BraTS-scale; img_size must match PATCH_SIZE (96³)
# SW_ROI=128³ at inference is fine — SwinUNETR uses sliding window, img_size
# only constrains the patch fed during training (must be divisible by 32).
SWIN_FEATURE_SIZE = 48
SWIN_DEPTHS       = (2, 2, 2, 2)
SWIN_NUM_HEADS    = (3, 6, 12, 24)
SWIN_DROP_RATE    = 0.0
SWIN_ATTN_DROP    = 0.0
SWIN_DROP_PATH    = 0.0

# ── Training ──────────────────────────────────────────────────────────────────
MAX_EPOCHS          = 50
VAL_INTERVAL        = 2
BATCH_SIZE          = 2 * max(NUM_GPUS, 1)
VAL_BATCH_SIZE      = 1
LEARNING_RATE       = 1e-4
WEIGHT_DECAY        = 1e-5
USE_AMP             = True
CACHE_RATE          = 0.5
EARLY_STOP_PATIENCE = 40

DL_NUM_WORKERS    = 0
CACHE_NUM_WORKERS = 0
BEST_MODEL_NAME   = "best_swinunetr_et.pth"

# ── Loss hyperparameters ──────────────────────────────────────────────────────
# FTL: α=0.3 (low FP penalty), β=0.7 (high FN penalty) — missing ET is critical
# γ=0.75: focal exponent on Tversky — concentrates gradient on hard boundary voxels
# FocalDice γ=3.0: stronger focal term for sharp ring-enhancing boundary
# ET_POS_WEIGHT=30: BCE weight on ET voxels — forces gradient when ET is rare
FTL_ALPHA    = 0.3
FTL_BETA     = 0.7
FTL_GAMMA    = 0.75

FDL_GAMMA    = 3.0       # Focal Dice exponent
ET_POS_WEIGHT = 30.0     # BCE positive weight on ET voxels

# Blending: FTL handles imbalance, FocalDice sharpens boundary, BCE prevents collapse
COMBINED_FTL_WEIGHT = 0.50
COMBINED_FDL_WEIGHT = 0.30
COMBINED_BCE_WEIGHT = 0.20

print(f"Input channels  : {IN_CHANNELS}  {IMAGE_KEYS}")
print(f"Output channels : {OUT_CHANNELS}  (BG=0, ET=1)")
print(f"ET primary keys : {ET_KEYS}  ← intensity augmentation applied here")
print(f"FTL params      : alpha={FTL_ALPHA}, beta={FTL_BETA}, gamma={FTL_GAMMA}")
print(f"FocalDice gamma : {FDL_GAMMA}")
print(f"ET pos weight   : {ET_POS_WEIGHT}")
print(f"Loss blend      : FTL={COMBINED_FTL_WEIGHT} + FocalDice={COMBINED_FDL_WEIGHT} + BCE={COMBINED_BCE_WEIGHT}")
print(f"Device          : {DEVICE}")


Run directory: C:\Users\BRAVO 15\OneDrive\Desktop\Intern\model_code\BraTS_2026\SwinUNETR_ET_BinarySubregion\run_20260616_1206
Input channels  : 3  ['t1n', 't1c', 't2w']
Output channels : 2  (BG=0, ET=1)
ET primary keys : ['t1c', 't1n']  ← intensity augmentation applied here
FTL params      : alpha=0.3, beta=0.7, gamma=0.75
FocalDice gamma : 3.0
ET pos weight   : 30.0
Loss blend      : FTL=0.5 + FocalDice=0.3 + BCE=0.2
Device          : cpu


## Cell 3 — Custom Modules: SwinUNETR Wrapper + Triple-Term ET Loss

### Why SwinUNETR for ET?
ET has a sharp, ring-enhancing boundary where long-range spatial context matters for delineating the full tumour rim.
- **Shifted-window self-attention** captures dependencies across the full 96³ patch, enabling the model to see the whole ring in context rather than just local patches.
- **Hierarchical feature encoding** at multiple scales provides both fine-grained boundary detail and global tumour shape awareness.

### Why the Triple-Term Loss?
- **Focal Tversky Loss (FTL)**: Penalises False Negatives (β=0.7 > α=0.3) — critical because missing ET is clinically far worse than over-predicting background. γ=0.75 focal exponent concentrates gradient on hard uncertain voxels.
- **Focal Dice Loss**: γ=3.0 punishes failures on difficult ring-boundary voxels rather than the easy background.
- **Weighted BCE**: pos_weight=30 ensures a non-zero gradient toward ET even at very low ET density (some paediatric cases have <0.1% ET voxels).


In [13]:
# ── SwinUNETR wrapper for ET ──────────────────────────────────────────────────
# Replaces CBAMUNet3D — identical call interface used in Cells 8, 11, 12.
# SwinUNETR shifted-window attention captures long-range context across the full
# 96³ patch, well-suited for the ring-enhancing tumour boundary of ET.
#
# img_size MUST match PATCH_SIZE (96³) so partition windows tile evenly.
# feature_size=48 → ~62 M params; use_checkpoint=True saves VRAM during training.

class SwinUNETRWrapper(nn.Module):
    """
    Thin wrapper around MONAI SwinUNETR exposing the same __init__ signature
    as CBAMUNet3D, for drop-in compatibility across all notebook cells.
    """
    def __init__(
        self,
        in_channels: int = 4,
        out_channels: int = 2,
        img_size=None,
        feature_size: int = 48,
        depths=(2, 2, 2, 2),
        num_heads=(3, 6, 12, 24),
        drop_rate: float = 0.0,
        attn_drop_rate: float = 0.0,
        dropout_path_rate: float = 0.0,
        use_checkpoint: bool = True,
    ):
        super().__init__()
        if img_size is None:
            img_size = PATCH_SIZE
        self.net = SwinUNETR(
            #img_size           = img_size,
            in_channels        = in_channels,
            out_channels       = out_channels,
            feature_size       = feature_size,
            depths             = depths,
            num_heads          = num_heads,
            drop_rate          = drop_rate,
            attn_drop_rate     = attn_drop_rate,
            dropout_path_rate  = dropout_path_rate,
            use_checkpoint     = use_checkpoint,
            spatial_dims       = 3,
        )

    def forward(self, x):
        return self.net(x)


# Sanity check
_sw = SwinUNETRWrapper(
    in_channels=3, out_channels=2,
    img_size=PATCH_SIZE, feature_size=SWIN_FEATURE_SIZE,
    depths=SWIN_DEPTHS, num_heads=SWIN_NUM_HEADS,
)
with torch.no_grad():
    _out = _sw(torch.randn(1, 3, *PATCH_SIZE))
assert _out.shape == (1, 2, *PATCH_SIZE), f"Unexpected shape: {_out.shape}"
del _sw, _out
print(f"SwinUNETRWrapper sanity check passed ✓  (img_size={PATCH_SIZE})")


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Focal Tversky Loss                                                          ║
# ║  Abraham & Khan, 2019 — adapted for BraTS binary ET segmentation            ║
# ║  TI = TP / (TP + α·FP + β·FN)   FTL = (1 - TI)^γ                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class FocalTverskyLoss(nn.Module):
    """
    Focal Tversky Loss for binary segmentation.

    α=0.3, β=0.7: Tolerates more FP (over-prediction) to avoid missing ET.
                  Missing ET (FN) is penalised 2.3× harder than over-predicting BG.
    γ=0.75: Focuses gradient on hard uncertain boundary voxels.
    """
    def __init__(self, alpha: float = 0.3, beta: float = 0.7,
                 gamma: float = 0.75, smooth: float = 1e-6):
        super().__init__()
        assert abs(alpha + beta - 1.0) < 1e-6, "alpha + beta must equal 1.0"
        self.alpha  = alpha
        self.beta   = beta
        self.gamma  = gamma
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        logits  : (B, 2, D, H, W)  — raw 2-class logits
        targets : (B, 1, D, H, W)  — integer labels {0, 1}
        """
        probs = F.softmax(logits, dim=1)[:, 1]       # (B, D, H, W) — ET probability
        tgt   = (targets[:, 0] == 1).float()          # (B, D, H, W) — binary ET mask

        probs = probs.contiguous().view(probs.size(0), -1)
        tgt   = tgt.contiguous().view(tgt.size(0), -1)

        TP = (probs * tgt).sum(dim=1)
        FP = (probs * (1.0 - tgt)).sum(dim=1)
        FN = ((1.0 - probs) * tgt).sum(dim=1)

        tversky_index = (TP + self.smooth) / (
            TP + self.alpha * FP + self.beta * FN + self.smooth
        )
        ftl = (1.0 - tversky_index) ** self.gamma
        return ftl.mean()


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Triple-Term ET Loss: FTL + FocalDice + WeightedBCE                         ║
# ║                                                                              ║
# ║  Why three terms?                                                            ║
# ║  1. FTL: overlap-based, imbalance-robust, penalises FN heavily              ║
# ║  2. FocalDice: sharpens boundary by down-weighting easy BG voxels           ║
# ║  3. WeightedBCE: prevents all-background collapse for sparse ET cases       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class CombinedETLoss(nn.Module):
    """
    Triple-term loss for ET binary segmentation.

    Args:
        ftl_alpha, ftl_beta, ftl_gamma : Focal Tversky parameters
        fdl_gamma                      : Focal Dice exponent (default 3.0)
        pos_weight                     : BCE positive weight on ET voxels
        ftl_weight, fdl_weight,
        bce_weight                     : Blending coefficients (must sum ≤ 1)
        smooth                         : Laplace smoothing
    """
    def __init__(self,
                 ftl_alpha: float = 0.3, ftl_beta: float = 0.7, ftl_gamma: float = 0.75,
                 fdl_gamma: float = 3.0, pos_weight: float = 30.0,
                 ftl_weight: float = 0.50, fdl_weight: float = 0.30,
                 bce_weight: float = 0.20, smooth: float = 1e-6):
        super().__init__()
        self.ftl_weight = ftl_weight
        self.fdl_weight = fdl_weight
        self.bce_weight = bce_weight
        self.fdl_gamma  = fdl_gamma
        self.smooth     = smooth

        self.focal_tversky = FocalTverskyLoss(
            alpha=ftl_alpha, beta=ftl_beta, gamma=ftl_gamma, smooth=smooth
        )
        self.register_buffer("pw", torch.tensor(pos_weight))

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=1)[:, 1]      # (B, D, H, W) — ET probability
        tgt   = (targets[:, 0] == 1).float()         # (B, D, H, W) — binary ET mask

        flat_p = probs.contiguous().view(probs.size(0), -1)
        flat_t = tgt.contiguous().view(tgt.size(0), -1)

        # ── Term 1: Focal Tversky Loss ──────────────────────────────────────
        ftl_val = self.focal_tversky(logits, targets)

        # ── Term 2: Focal Dice Loss ─────────────────────────────────────────
        intersection = (flat_p * flat_t).sum(dim=1)
        dice_coef    = (2.0 * intersection + self.smooth) / (
            flat_p.sum(dim=1) + flat_t.sum(dim=1) + self.smooth
        )
        focal_dice = ((1.0 - dice_coef) ** self.fdl_gamma).mean()

        # ── Term 3: Weighted BCE ────────────────────────────────────────────
        flat_p_c = flat_p.clamp(1e-7, 1.0 - 1e-7)
        bce_raw  = -(
            self.pw * flat_t * torch.log(flat_p_c)
            + (1.0 - flat_t) * torch.log(1.0 - flat_p_c)
        )
        weighted_bce = bce_raw.mean(dim=1).mean()

        return (self.ftl_weight * ftl_val
                + self.fdl_weight * focal_dice
                + self.bce_weight * weighted_bce)


# Sanity checks
_loss_fn = CombinedETLoss(
    ftl_alpha=FTL_ALPHA, ftl_beta=FTL_BETA, ftl_gamma=FTL_GAMMA,
    fdl_gamma=FDL_GAMMA, pos_weight=ET_POS_WEIGHT,
    ftl_weight=COMBINED_FTL_WEIGHT, fdl_weight=COMBINED_FDL_WEIGHT,
    bce_weight=COMBINED_BCE_WEIGHT,
)
_lv = _loss_fn(torch.randn(2, 2, 8, 8, 8), torch.randint(0, 2, (2, 1, 8, 8, 8)))
print(f"CombinedETLoss sanity check: {_lv.item():.4f} ✓")
print(f"  FTL  α={FTL_ALPHA} β={FTL_BETA} γ={FTL_GAMMA}  weight={COMBINED_FTL_WEIGHT}")
print(f"  FDL  γ={FDL_GAMMA}                              weight={COMBINED_FDL_WEIGHT}")
print(f"  BCE  pos_weight={ET_POS_WEIGHT}                        weight={COMBINED_BCE_WEIGHT}")


SwinUNETRWrapper sanity check passed ✓  (img_size=(64, 64, 64))
CombinedETLoss sanity check: 3.2235 ✓
  FTL  α=0.3 β=0.7 γ=0.75  weight=0.5
  FDL  γ=3.0                              weight=0.3
  BCE  pos_weight=30.0                        weight=0.2


## Cell 4 — Data Discovery
ET binary label: original label 1 → 1, all others → 0.
All cases are included — cases with no ET teach the model specificity (do not predict ET where absent).


In [14]:
FILE_KEYS = ["t1n", "t1c", "t2w", "t2f"]


def find_modality(case_dir: Path, key: str):
    # Try looking for the standard pattern first
    hits = sorted(case_dir.glob(f"*-{key}.nii.gz"))
    if not hits:
        # Grab files containing the key, but FILTER OUT any mask files
        hits = sorted([
            p for p in case_dir.glob(f"*{key}*.nii*") 
            if "mask" not in p.name.lower()
        ])
    return str(hits[0]) if hits else None
data_dicts = []
skipped    = []

case_dirs = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir()])

for d in case_dirs:
    sid = d.name
    mod_paths = {key: find_modality(d, key) for key in FILE_KEYS}
    missing   = [k for k, v in mod_paths.items() if v is None]
    if missing:
        skipped.append((sid, f"missing modalities: {missing}"))
        continue

    seg_hits = sorted(d.glob("*-seg.nii.gz")) or sorted(d.glob("*seg*.nii*"))
    if not seg_hits:
        skipped.append((sid, "no seg file"))
        continue

    # Include ALL cases — model must learn to not predict ET where absent
    entry = {key: mod_paths[key] for key in FILE_KEYS}
    entry["label"]      = str(seg_hits[0])
    entry["subject_id"] = sid
    data_dicts.append(entry)

print(f"Total cases found : {len(data_dicts)}")
print(f"Skipped           : {len(skipped)}")

from collections import Counter
reasons = Counter(reason for _, reason in skipped)
for reason, count in reasons.most_common():
    print(f"  {count:4d}  {reason}")

# Count ET prevalence
et_count    = 0
no_et_count = 0
for entry in data_dicts:
    seg_arr = nib.load(entry["label"]).get_fdata()
    if (seg_arr == 1).sum() > 0:
        et_count += 1
    else:
        no_et_count += 1

print(f"\nCases WITH    ET (label 1 present) : {et_count}")
print(f"Cases WITHOUT ET (label 1 absent)  : {no_et_count}")
print(f"Total                              : {len(data_dicts)}")

# Label sanity on first case
if data_dicts:
    seg_img = nib.load(data_dicts[0]["label"])
    seg_arr = seg_img.get_fdata()
    print(f"\nLabel sanity check — first case: {data_dicts[0]['subject_id']}")
    full_map = {0: "BG", 1: "ET", 2: "NET", 3: "CC", 4: "ED"}
    for lbl in np.unique(seg_arr).astype(int):
        name = full_map.get(lbl, "UNKNOWN")
        vox  = int((seg_arr == lbl).sum())
        print(f"  Label {lbl} ({name:>3s}): {vox:>10,} voxels → binary ET: {1 if lbl==1 else 0}")


Total cases found : 20
Skipped           : 0

Cases WITH    ET (label 1 present) : 15
Cases WITHOUT ET (label 1 absent)  : 5
Total                              : 20

Label sanity check — first case: BraTS-PED-00010-000
  Label 0 ( BG):  8,828,392 voxels → binary ET: 0
  Label 1 ( ET):      4,280 voxels → binary ET: 1
  Label 2 (NET):     95,328 voxels → binary ET: 0


## Cell 5 — Transforms

**ET-specific design choices:**
- **Binary remap**: label 1 → 1, all others → 0
- **T1c as foreground crop source** — ET ring-enhancement is uniquely bright on T1CE
- **pos=4, neg=1** — 80% of patches forced onto ET voxels (critical for sparse ET)
- **Gamma correction on ET_KEYS (t1c, t1n)** — simulates contrast-uptake variability between scanners
- **RandAdjustContrast on T1c/T1n** — mimics variation in gadolinium dose and timing
- `allow_smaller=True` — prevents crash when tumour is near image boundary


In [34]:
from monai.transforms import Lambdad

# ── Custom: Gamma correction for T1CE intensity variability ───────────────────
class RandGammaCorrectiond(MapTransform):
    """
    Per-channel gamma correction on selected modality keys.
    Models contrast-uptake intensity variation in T1c across scanners/protocols.
    gamma < 1: brightens image (increases contrast enhancement visibility)
    gamma > 1: darkens image (simulates reduced gadolinium uptake)
    """
    def __init__(self, keys, gamma_range=(0.7, 1.5), prob=0.5,
                 allow_missing_keys=False):
        super().__init__(keys, allow_missing_keys)
        self.gamma_range = gamma_range
        self.prob        = prob

    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            if random.random() < self.prob:
                gamma   = random.uniform(*self.gamma_range)
                arr     = d[key]
                if isinstance(arr, torch.Tensor):
                    arr_min = arr.min()
                    arr_max = arr.max()
                    if arr_max > arr_min:
                        norm  = (arr - arr_min) / (arr_max - arr_min + 1e-8)
                        arr   = norm.pow(gamma) * (arr_max - arr_min) + arr_min
                else:
                    arr_min = arr.min()
                    arr_max = arr.max()
                    if arr_max > arr_min:
                        norm  = (arr - arr_min) / (arr_max - arr_min + 1e-8)
                        arr   = norm ** gamma * (arr_max - arr_min) + arr_min
                d[key] = arr
        return d


# ── Key groups ─────────────────────────────────────────────────────────────────
IMAGE_KEYS = ["t1n", "t1c", "t2w","t2f"]
ET_KEYS    = ["t2f", "t2w"]   # Gamma + contrast augmentation on T1CE pair
ALL_KEYS   = IMAGE_KEYS + ["label"]

IN_CHANNELS  = 4
OUT_CHANNELS = 2

print(f"IMAGE_KEYS : {IMAGE_KEYS}  ({IN_CHANNELS} channels into model)")
print(f"ET_KEYS    : {ET_KEYS}  ← gamma correction + contrast augmentation")
print(f"ALL_KEYS   : {ALL_KEYS}")


# ── Train transforms ───────────────────────────────────────────────────────────
train_transforms = Compose([

    # ── Load & reorient ───────────────────────────────────────────────────────
    LoadImaged(keys=ALL_KEYS),
    EnsureChannelFirstd(keys=ALL_KEYS),
    Orientationd(keys=ALL_KEYS, axcodes="RAS"),

    # ── Label remap BEFORE crop ───────────────────────────────────────────────
    # pos/neg sampler must see binary {0,1}, not 5-class
    #RemapETLabel(keys=["label"]),
    Lambdad(keys=["label"], func=lambda x: (x == 1).long()),
    Spacingd(keys=ALL_KEYS,pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest"), padding_mode="border"),
    # ── Normalize all modalities ──────────────────────────────────────────────
    NormalizeIntensityd(keys=IMAGE_KEYS, nonzero=True, channel_wise=True),

    # ── Foreground crop using T1c ─────────────────────────────────────────────
    # T1c source: ET ring-enhancement is UNIQUELY BRIGHT on post-contrast T1.
    # margin=10 ensures cropped volume always exceeds 96³ patch size.
    CropForegroundd(
        keys=ALL_KEYS,
        source_key="t1c",
        select_fn=lambda x: x > 0,
        allow_smaller=True,
        margin=10,
    ),

    # ── Patch sampling biased toward ET ──────────────────────────────────────
    # pos=4: ~80% of patches forced onto ET voxels — critical for sparse ET.
    # image_key="t1c": mask uses T1c non-zero region (enhancing tumour territory).
    RandCropByPosNegLabeld(
        keys=ALL_KEYS,
        label_key="label",
        spatial_size=list(PATCH_SIZE),
        pos=3,
        neg=1,
        num_samples=NUM_POS_NEG_SAMPLES,
        image_key="t1c",
        image_threshold=0,
        allow_smaller=True,
    ),

    # ── Spatial augmentations ─────────────────────────────────────────────────
    RandFlipd(keys=ALL_KEYS, prob=0.5, spatial_axis=0),
    RandFlipd(keys=ALL_KEYS, prob=0.5, spatial_axis=1),
    RandFlipd(keys=ALL_KEYS, prob=0.5, spatial_axis=2),
    RandRotate90d(keys=ALL_KEYS, prob=0.5, max_k=3),

    #RandAffined
    #RandZoomd
    # ── ED-SPECIFIC: T2/FLAIR intensity augmentations ─────────────────────────
    # Heavier augmentation on T2w and T2f only (primary ED channels)
    # Simulates scanner variability in fluid-tissue contrast
    #RandGaussianNoised(keys=IMAGE_KEYS, std=0.02, mean=0.0,prob=0.25),       # ↑ from 0.01
    # Gaussian blur on T2/FLAIR: mimics partial volume effects at edema boundary
    #RandGaussianSmoothd(keys=IMAGE_KEYS,sigma_x=(0.5, 1.5),sigma_y=(0.5, 1.5),sigma_z=(0.5, 1.5), prob=0.2),
    RandAdjustContrastd(keys=ET_KEYS, prob=0.3, gamma=(0.7, 1.5),),
    RandScaleIntensityd(keys=IMAGE_KEYS, factors=0.2, prob=0.4),
    RandShiftIntensityd(keys=IMAGE_KEYS, offsets=0.1, prob=0.4),
    #RandGibbsNoised
    #RandGammaCorrectiond(keys=ED_KEYS, gamma_range=(0.7, 1.4), prob=0.5),
])


# ── Val / test transforms ──────────────────────────────────────────────────────
val_transforms = Compose([
    LoadImaged(keys=ALL_KEYS),
    EnsureChannelFirstd(keys=ALL_KEYS),
    Orientationd(keys=ALL_KEYS, axcodes="RAS"),
    #RemapETLabel(keys=["label"]),
    Lambdad(keys=["label"], func=lambda x: (x == 1).long()),
    NormalizeIntensityd(keys=IMAGE_KEYS, nonzero=True, channel_wise=True),
    CropForegroundd(
        keys=ALL_KEYS,
        source_key="t1n",
        select_fn=lambda x: x > 0,
        allow_smaller=True,
        margin=15,
    ),
])
test_transforms = val_transforms


print("\nTransforms defined ✓")
print(f"  Train steps : {len(train_transforms.transforms)}")
print(f"  Val steps   : {len(val_transforms.transforms)}")
print(f"  Crop source : t2f (ET is T1CE-hyperintense, ring-enhancing)")
print(f"  Patch size  : {PATCH_SIZE}  pos:neg = 4:1")
print(f"  ET augments : Gamma correction + RandAdjustContrast on T1c/T1n")


IMAGE_KEYS : ['t1n', 't1c', 't2w']  (3 channels into model)
ET_KEYS    : ['t1c', 't1n']  ← gamma correction + contrast augmentation
ALL_KEYS   : ['t1n', 't1c', 't2w', 'label']

Transforms defined ✓
  Train steps : 17
  Val steps   : 6
  Crop source : t2f (ET is T1CE-hyperintense, ring-enhancing)
  Patch size  : (64, 64, 64)  pos:neg = 4:1
  ET augments : Gamma correction + RandAdjustContrast on T1c/T1n


## Cell 6 — Train / Val / Test Split

In [36]:
rng     = np.random.default_rng(SEED)
indices = np.arange(len(data_dicts))
rng.shuffle(indices)

n_total = len(indices)
n_train = int(round(n_total * SPLIT_RATIOS[0]))
n_val   = int(round(n_total * SPLIT_RATIOS[1]))

train_files = [data_dicts[i] for i in indices[:n_train]]
val_files   = [data_dicts[i] for i in indices[n_train : n_train + n_val]]
test_files  = [data_dicts[i] for i in indices[n_train + n_val:]]

splits_df = pd.DataFrame(
    [{"subject_id": d["subject_id"], "split": s}
     for files, s in [(train_files,"train"),(val_files,"val"),(test_files,"test")]
     for d in files]
).sort_values("subject_id").reset_index(drop=True)
splits_df.to_csv(RUN_DIR / "splits.csv", index=False)
print(f"train={len(train_files)}  val={len(val_files)}  test={len(test_files)}")


train=16  val=4  test=0


## Cell 7 — Datasets & DataLoaders

In [37]:
train_ds = SmartCacheDataset(
    data=train_files, transform=train_transforms,
    cache_rate=CACHE_RATE,
    num_init_workers=CACHE_NUM_WORKERS,
    num_replace_workers=CACHE_NUM_WORKERS,
)
val_ds = SmartCacheDataset(
    data=val_files, transform=val_transforms,
    cache_rate=CACHE_RATE,
    num_init_workers=CACHE_NUM_WORKERS,
    num_replace_workers=CACHE_NUM_WORKERS,
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=DL_NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=False,
)
val_loader = DataLoader(
    val_ds, batch_size=VAL_BATCH_SIZE, shuffle=False,
    num_workers=DL_NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=False, collate_fn=pad_list_data_collate,
)
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")


Loading dataset: 100%|██████████| 2/2 [00:01<00:00,  1.68it/s]

Train batches : 4
Val batches   : 2


## Cell 8 — Model, Loss, Optimiser, Metrics
- **SwinUNETR**: Shifted-window transformer (feature_size=48, img_size=96³)
- **CombinedETLoss**: FTL(α=0.3,β=0.7,γ=0.75) + FocalDice(γ=3) + WeightedBCE(pos_weight=30)
- **out_channels=2** (BG=0, ET=1)


In [38]:
# ── Model ─────────────────────────────────────────────────────────────────────
model = SwinUNETRWrapper(
    in_channels       = IN_CHANNELS,
    out_channels      = OUT_CHANNELS,
    img_size          = PATCH_SIZE,
    feature_size      = SWIN_FEATURE_SIZE,
    depths            = SWIN_DEPTHS,
    num_heads         = SWIN_NUM_HEADS,
    drop_rate         = SWIN_DROP_RATE,
    attn_drop_rate    = SWIN_ATTN_DROP,
    dropout_path_rate = SWIN_DROP_PATH,
    use_checkpoint    = True,
).to(DEVICE)

if NUM_GPUS > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel across {NUM_GPUS} GPUs")

try:
    model = torch.compile(model, backend="aot_eager")
    print("torch.compile applied (aot_eager)")
except Exception as e:
    print(f"torch.compile skipped: {e}")

n_params = sum(p.numel() for p in model.parameters())
print(f"SwinUNETR parameters: {n_params/1e6:.2f} M")

# ── Loss ──────────────────────────────────────────────────────────────────────
loss_function = CombinedETLoss(
    ftl_alpha   = FTL_ALPHA,
    ftl_beta    = FTL_BETA,
    ftl_gamma   = FTL_GAMMA,
    fdl_gamma   = FDL_GAMMA,
    pos_weight  = ET_POS_WEIGHT,
    ftl_weight  = COMBINED_FTL_WEIGHT,
    fdl_weight  = COMBINED_FDL_WEIGHT,
    bce_weight  = COMBINED_BCE_WEIGHT,
).to(DEVICE)

# ── Post-processing ───────────────────────────────────────────────────────────
post_label = AsDiscrete(to_onehot=OUT_CHANNELS)
post_pred  = AsDiscrete(argmax=True, to_onehot=OUT_CHANNELS)

# ── Metrics ───────────────────────────────────────────────────────────────────
dice_metric = DiceMetric(
    include_background=False,
    reduction=MetricReduction.MEAN_BATCH,
    get_not_nans=True,
)
hd95_metric = HausdorffDistanceMetric(
    include_background=False, percentile=95.0,
    reduction=MetricReduction.MEAN_BATCH,
)

def _get_raw(m):
    """Unwrap torch.compile / DataParallel to get raw state dict."""
    if hasattr(m, "_orig_mod"): m = m._orig_mod
    if hasattr(m, "module"):    m = m.module
    return m

# ── Optimiser ─────────────────────────────────────────────────────────────────
try:
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY, fused=True,
    )
    print("Fused AdamW applied")
except TypeError:
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    print("Standard AdamW")

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS,
)

# ── Sliding-window inferer ────────────────────────────────────────────────────
model_inferer = partial(
    sliding_window_inference,
    roi_size=SW_ROI, sw_batch_size=SW_BATCH,
    predictor=model, overlap=SW_OVERLAP,
)

# ── GradScaler ────────────────────────────────────────────────────────────────
scaler = GradScaler(enabled=USE_AMP and torch.cuda.is_available())

print("\nAll components ready ✓")
print(f"  Loss   : CombinedETLoss  FTL({FTL_ALPHA}/{FTL_BETA}/{FTL_GAMMA}) + FDL({FDL_GAMMA}) + BCE({ET_POS_WEIGHT})")
print(f"  Output : {OUT_CHANNELS} channels → BG(0) | ET(1)")


torch.compile applied (aot_eager)
SwinUNETR parameters: 62.19 M
Fused AdamW applied

All components ready ✓
  Loss   : CombinedETLoss  FTL(0.3/0.7/0.75) + FDL(3.0) + BCE(30.0)
  Output : 2 channels → BG(0) | ET(1)


## Cell 9 — Training Loop
Saves checkpoint on best **ET Dice**. Early stops after no improvement for `EARLY_STOP_PATIENCE` validation intervals.

In [39]:
best_metric              = -1.0
best_metric_epoch        = -1
epochs_since_improvement = 0
epoch_loss_values        = []
val_metric_values        = []
log_rows                 = []
run_start                = time.time()
cc_dice                  = float("nan")   # initialise for log on non-val epochs

# ── Progress widgets ──────────────────────────────────────────────────────────
epoch_bar    = widgets.IntProgress(
    value=0, min=0, max=MAX_EPOCHS,
    description="Epochs:", bar_style="info",
    layout=widgets.Layout(width="80%"),
)
status_label = widgets.Label(value="Starting...")
val_label    = widgets.Label(value="")
display(widgets.VBox([epoch_bar, status_label, val_label]))

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(1, MAX_EPOCHS + 1):

    # ── TRAIN ─────────────────────────────────────────────────────────────────
    model.train()
    epoch_loss  = 0.0
    n_steps     = 0
    epoch_start = time.time()

    for batch_data in train_loader:
        inputs = torch.cat(
            [batch_data[k].to(DEVICE, non_blocking=True) for k in IMAGE_KEYS],
            dim=1,
        )
        labels = batch_data["label"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=USE_AMP):
            outputs = model(inputs)
            loss    = loss_function(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        n_steps    += 1

    epoch_loss /= max(n_steps, 1)
    epoch_loss_values.append(epoch_loss)
    scheduler.step()

    elapsed = time.time() - epoch_start
    lr_now  = optimizer.param_groups[0]["lr"]

    epoch_bar.value    = epoch
    status_label.value = (
        f"Epoch {epoch}/{MAX_EPOCHS}  |  Loss={epoch_loss:.4f}  "
        f"|  lr={lr_now:.2e}  |  {elapsed:.1f}s"
    )
    print(f"Epoch {epoch:03d}/{MAX_EPOCHS} | Loss: {epoch_loss:.4f} | lr: {lr_now:.2e} | {elapsed:.1f}s")

    # ── VALIDATE ──────────────────────────────────────────────────────────────
    if epoch % VAL_INTERVAL == 0:
        model.eval()
        val_start = time.time()

        with torch.no_grad():
            for vb in val_loader:
                vi = torch.cat(
                    [vb[k].to(DEVICE, non_blocking=True) for k in IMAGE_KEYS],
                    dim=1,
                )
                vl = vb["label"].to(DEVICE, non_blocking=True)

                with autocast(enabled=USE_AMP):
                    vo_logits = model_inferer(vi)

                vo_list = decollate_batch(vo_logits)
                vl_list = decollate_batch(vl)
                vo_bin  = [post_pred(p)  for p in vo_list]
                vl_bin  = [post_label(l) for l in vl_list]
                dice_metric(y_pred=vo_bin, y=vl_bin)

        dice_agg, _ = dice_metric.aggregate()
        dice_metric.reset()
        et_dice = float(dice_agg.mean().item())
        val_metric_values.append({"epoch": epoch, "ET_dice": et_dice})

        improved_marker = ""
        if et_dice > best_metric:
            best_metric              = et_dice
            best_metric_epoch        = epoch
            epochs_since_improvement = 0
            raw = _get_raw(model)
            torch.save(raw.state_dict(), RUN_DIR / BEST_MODEL_NAME)
            torch.save(
                raw.state_dict(),
                RUN_DIR / "checkpoints" / f"et_epoch{epoch:03d}_dice{et_dice:.4f}.pth"
            )
            improved_marker     = "  *** NEW BEST ***"
            epoch_bar.bar_style = "success"
        else:
            epochs_since_improvement += 1
            epoch_bar.bar_style = "info"

        val_elapsed = time.time() - val_start
        val_label.value = (
            f"Val @ ep {epoch}  |  ET Dice={et_dice:.4f}  "
            f"|  best={best_metric:.4f}@ep{best_metric_epoch}  "
            f"|  {val_elapsed:.1f}s{improved_marker}"
        )
        print(f" ---> [VAL] ET Dice: {et_dice:.4f} | Best: {best_metric:.4f} @ Ep {best_metric_epoch}{improved_marker}")

        # Early stopping
        if (EARLY_STOP_PATIENCE is not None
                and epochs_since_improvement >= EARLY_STOP_PATIENCE):
            print(f"\nEarly stop @ epoch {epoch} — best ET Dice={best_metric:.4f} @ ep {best_metric_epoch}")
            epoch_bar.bar_style = "warning"
            break

    # ── Save log every epoch ──────────────────────────────────────────────────
    log_rows.append({
        "epoch":         epoch,
        "lr":            lr_now,
        "train_loss":    epoch_loss,
        "val_ET_dice":   et_dice if epoch % VAL_INTERVAL == 0 else float("nan"),
        "epoch_seconds": round(elapsed, 1),
    })
    pd.DataFrame(log_rows).to_csv(RUN_DIR / "training_log.csv", index=False)

total_min = (time.time() - run_start) / 60
status_label.value = (
    f"Done — {total_min:.1f} min  |  best ET Dice={best_metric:.4f} @ ep {best_metric_epoch}"
)
epoch_bar.bar_style = "success"
print(f"\nTraining complete — {total_min:.1f} min | Best ET Dice: {best_metric:.4f} @ epoch {best_metric_epoch}")


Epoch 001/50 | Loss: 0.9559 | lr: 9.99e-05 | 184.1s
Epoch 002/50 | Loss: 0.9201 | lr: 9.96e-05 | 179.1s
 ---> [VAL] ET Dice: 0.0086 | Best: 0.0086 @ Ep 2  *** NEW BEST ***
Epoch 003/50 | Loss: 0.8950 | lr: 9.91e-05 | 180.5s
Epoch 004/50 | Loss: 0.9031 | lr: 9.84e-05 | 180.4s
 ---> [VAL] ET Dice: 0.0211 | Best: 0.0211 @ Ep 4  *** NEW BEST ***
Epoch 005/50 | Loss: 0.8541 | lr: 9.76e-05 | 179.8s
Epoch 006/50 | Loss: 0.8400 | lr: 9.65e-05 | 179.5s
 ---> [VAL] ET Dice: 0.0230 | Best: 0.0230 @ Ep 6  *** NEW BEST ***
Epoch 007/50 | Loss: 0.8396 | lr: 9.52e-05 | 180.2s
Epoch 008/50 | Loss: 0.8214 | lr: 9.38e-05 | 180.9s
 ---> [VAL] ET Dice: 0.0307 | Best: 0.0307 @ Ep 8  *** NEW BEST ***
Epoch 009/50 | Loss: 0.8117 | lr: 9.22e-05 | 180.0s
Epoch 010/50 | Loss: 0.8354 | lr: 9.05e-05 | 179.6s
 ---> [VAL] ET Dice: 0.0389 | Best: 0.0389 @ Ep 10  *** NEW BEST ***
Epoch 011/50 | Loss: 0.8216 | lr: 8.85e-05 | 180.5s
Epoch 012/50 | Loss: 0.8042 | lr: 8.64e-05 | 183.6s
 ---> [VAL] ET Dice: 0.0451 | Best:

KeyboardInterrupt: 

## Cell 10 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(range(1, len(epoch_loss_values)+1), epoch_loss_values,
             lw=1.5, color="crimson")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Combined Loss")
axes[0].set_title("Training Loss — ET Subregion 3D UNet")
axes[0].grid(alpha=0.3)

if val_metric_values:
    val_df = pd.DataFrame(val_metric_values)
    axes[1].plot(val_df["epoch"], val_df["ET_dice"], "o-",
                 lw=1.5, color="crimson", label="ET Dice")
    if best_metric_epoch > 0:
        axes[1].axvline(best_metric_epoch, ls="--", c="black", alpha=0.5,
                        label=f"best={best_metric:.3f}@ep{best_metric_epoch}")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Dice")
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Val ET Dice (Binary: BG vs ET)")
    axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
out_path = RUN_DIR / "figures" / "training_curves_et.png"
plt.savefig(out_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")


## Cell 11 — Test-Set Evaluation

In [ ]:
dice_metric_t = DiceMetric(
    include_background=False, reduction=MetricReduction.MEAN_BATCH, get_not_nans=True,
)
hd95_metric_t = HausdorffDistanceMetric(
    include_background=False, percentile=95.0, reduction=MetricReduction.MEAN_BATCH,
)

test_ds     = Dataset(data=test_files, transform=test_transforms)
test_loader = DataLoader(
    test_ds, batch_size=1, shuffle=False,
    num_workers=DL_NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=False, collate_fn=pad_list_data_collate,
)

# Load best checkpoint into a fresh (unwrapped) model
raw_model = SwinUNETRWrapper(
    in_channels       = IN_CHANNELS,
    out_channels      = OUT_CHANNELS,
    img_size          = PATCH_SIZE,
    feature_size      = SWIN_FEATURE_SIZE,
    depths            = SWIN_DEPTHS,
    num_heads         = SWIN_NUM_HEADS,
    use_checkpoint    = False,   # no gradient checkpointing at inference
).to(DEVICE)
raw_model.load_state_dict(torch.load(RUN_DIR / BEST_MODEL_NAME, map_location=DEVICE))
raw_model.eval()

test_inferer = partial(
    sliding_window_inference,
    roi_size=SW_ROI, sw_batch_size=SW_BATCH,
    predictor=raw_model, overlap=SW_OVERLAP,
)

per_subject = []
print("Running ET test-set evaluation...")
print(f"{'Idx':>4}  {'Subject':<25}  {'ET Dice':>8}  {'HD95':>8}")
print("-" * 55)

with torch.no_grad():
    for idx, tb in enumerate(test_loader, 1):
        sid = tb["subject_id"][0]
        ti  = torch.cat([tb[k].to(DEVICE) for k in IMAGE_KEYS], dim=1)
        tl  = tb["label"].to(DEVICE)

        with autocast(enabled=USE_AMP):
            to_logits = test_inferer(ti)

        to_list = decollate_batch(to_logits)
        tl_list = decollate_batch(tl)
        to_bin  = [post_pred(p)  for p in to_list]
        tl_bin  = [post_label(l) for l in tl_list]

        dice_metric_t.reset()
        dice_metric_t(y_pred=to_bin, y=tl_bin)
        et_d = float(dice_metric_t.aggregate()[0].mean().item())

        try:
            hd95_metric_t.reset()
            hd95_metric_t(y_pred=to_bin, y=tl_bin)
            hd95 = float(hd95_metric_t.aggregate().mean().item())
        except Exception:
            hd95 = float("nan")

        pred_label = to_bin[0].argmax(dim=0).cpu().numpy().astype(np.uint8)
        ref_path   = [f for f in test_files if f["subject_id"] == sid][0]["label"]
        ref_img    = nib.load(ref_path)
        nib.save(
            nib.Nifti1Image(pred_label, ref_img.affine),
            str(RUN_DIR / "predictions" / f"{sid}_ET_pred.nii.gz")
        )

        per_subject.append({"subject_id": sid, "dice_ET": et_d, "hd95_mm": hd95})
        print(f"{idx:4d}  {sid:<25}  {et_d:8.4f}  {hd95:8.2f}")

per_df = pd.DataFrame(per_subject).sort_values("dice_ET", ascending=False)
per_df.to_csv(RUN_DIR / "test_metrics_et.csv", index=False)
print(f"\nMean ET Dice  : {per_df['dice_ET'].mean():.4f}")
print(f"Median ET Dice: {per_df['dice_ET'].median():.4f}")
print(f"Saved: {RUN_DIR / 'test_metrics_et.csv'}")


## Cell 12 — Reload Instructions

In [ ]:
print("=" * 60)
print(f" Best ET Dice (val): {best_metric:.4f} @ epoch {best_metric_epoch}")
print(f" Best model saved : {RUN_DIR / BEST_MODEL_NAME}")
print("=" * 60)
print("""
To reload for inference:

  model = SwinUNETRWrapper(
      in_channels=4, out_channels=2,
      img_size=(96, 96, 96),
      feature_size=48,
      depths=(2, 2, 2, 2),
      num_heads=(3, 6, 12, 24),
      use_checkpoint=False,
  )
  model.load_state_dict(torch.load("best_swinunetr_et.pth"))
  model.eval()

Input : 4-channel MRI (T1n, T1c, T2w, T2f) normalised channel-wise
Output: Binary mask — 0=Background, 1=ET (Enhancing Tumour)
Loss  : FTL(α=0.3,β=0.7,γ=0.75) + FocalDice(γ=3) + WeightedBCE(pos_weight=30)
Arch  : SwinUNETR shifted-window transformer (feature_size=48)
""")
